## Data collection

Gets all of the synched properties in the /ToolItems directory

In [54]:
import pathlib as pl
import re
from utils import *

# Gets the path of every .cs file in the ToolItems directory
project_base_dir = pl.Path(r'E:\simulator\ASTROS2020-Simulator-Project\Assets')
toolitems_base_dir = project_base_dir / 'ToolItems'
toolitem_file_paths = list(toolitems_base_dir.rglob('*.cs'))

toolitem_properties_map: dict[str, list[str]] = {}
attribute_pattern = re.compile(r'\[GenNetCodeSync')

# Iterates and opens each c# file in tool_items_base_dir
for file_path in toolitem_file_paths:
    with open(file_path, 'r', encoding='Latin-1', errors='ignore') as file:
        # Reads the file line-by-line
        lines = file.readlines()
        for idx, line in enumerate(lines):
            if attribute_pattern.search(line):
                # Look forward to find the property/field declaration
                for next_line in lines[idx + 1:]:
                    stripped = next_line.strip()
                    # Skip empty lines, comments, and further attribute lines
                    if not stripped or stripped.startswith('//') or stripped.startswith('['):
                        continue
                    if file_path.name not in toolitem_properties_map:
                        toolitem_properties_map[file_path.name] = []
                    # toolitem_properties_map[file_path.name].append(stripped)           # appends the entire property declaraytion line
                    stripped = stripped.split()
                    if "new" in stripped:
                        stripped.remove("new")
                    toolitem_properties_map[file_path.name].append(stripped[2])  # appends only the property name        
                    break

print(toolitem_properties_map)
properties_found_count = get_property_count(toolitem_properties_map)
print(f'Found [{properties_found_count}] properties accross [{len(toolitem_properties_map.keys())}] different files')


{'LocalTacticalLine.cs': ['tacticalType'], 'LocalMovementRestrictiveArea.cs': ['type', 'parts', 'name', 'color', 'layerId', 'editingClientId'], 'LocalInterestPoint.cs': ['type', 'geoData'], 'LocalItinerary.cs': ['points'], 'LocalToolItem.cs': ['toolItemType', 'name', 'color', 'layerId', 'editingClientId', 'flags'], 'LocalRectangle.cs': ['renderType', 'geoData'], 'LocalMultipartPolygon.cs': ['parts', 'renderType'], 'LocalLine.cs': ['controlPoints', 'renderType', 'arrowheadType'], 'LocalAbstractEllipse.cs': ['renderType', 'geoData'], 'LocalArrow.cs': ['renderType', 'arrowData'], 'LocalMeasure.cs': ['name', 'period', 'emitter', 'observation'], 'LocalShapedMeasure.cs': ['type', 'color', 'layerId', 'editingClientId'], 'LocalAreaShapedMeasure.cs': ['points'], 'LocalLineShapedMeasure.cs': ['points'], 'LocalInterdictionCell.cs': ['cellType', 'geoCenter', 'geoSize', 'degrees', 'ceilingAltitudeInMeters', 'separationAltitudeInMeters'], 'LocalAirspaceZone.cs': ['zoneType', 'widthInMeters', 'points

Let's make that 60 properties to get an even number an also be in line with what the paper says. LocalAirspaceZone (3), and LocalInterdictionCell (6)

In [55]:
toolitem_properties_map.pop('LocalAirspaceZone.cs')
toolitem_properties_map.pop('LocalInterdictionCell.cs')

properties_found_count = get_property_count(toolitem_properties_map)
print(f'Found [{properties_found_count}] properties accross [{len(toolitem_properties_map.keys())}] different files')

Found [60] properties accross [21] different files


Now that we have all of the classes and their properties beeing synched, we can now get the metrcis in a more organized fashion. The metrics being gathered are:
1. Development effort
    - LOC written by **property**
    - Number of files that need to be modified by **class**
2. Coupling
    - Number of times that the local **class** references the net one
    - Number of times that the SRP is violated for each **property**
3. Quality
    - Percentage of duplicated code in a **class**
    - Cyclomatic complexity for each **class**
4. Proccess:
    - Compilation time

All fo the metrics will be collected in this [toolitem: metrics] dict

In [56]:
toolitem_metrics_map: dict[str, dict[str, any]] = {}

for toolitem, properties in toolitem_properties_map.items():
    toolitem_metrics_map[toolitem] = {}
    for prop in properties:
        toolitem_metrics_map[toolitem][prop] = {}

print(toolitem_metrics_map)

{'LocalTacticalLine.cs': {'tacticalType': {}}, 'LocalMovementRestrictiveArea.cs': {'type': {}, 'parts': {}, 'name': {}, 'color': {}, 'layerId': {}, 'editingClientId': {}}, 'LocalInterestPoint.cs': {'type': {}, 'geoData': {}}, 'LocalItinerary.cs': {'points': {}}, 'LocalToolItem.cs': {'toolItemType': {}, 'name': {}, 'color': {}, 'layerId': {}, 'editingClientId': {}, 'flags': {}}, 'LocalRectangle.cs': {'renderType': {}, 'geoData': {}}, 'LocalMultipartPolygon.cs': {'parts': {}, 'renderType': {}}, 'LocalLine.cs': {'controlPoints': {}, 'renderType': {}, 'arrowheadType': {}}, 'LocalAbstractEllipse.cs': {'renderType': {}, 'geoData': {}}, 'LocalArrow.cs': {'renderType': {}, 'arrowData': {}}, 'LocalMeasure.cs': {'name': {}, 'period': {}, 'emitter': {}, 'observation': {}}, 'LocalShapedMeasure.cs': {'type': {}, 'color': {}, 'layerId': {}, 'editingClientId': {}}, 'LocalAreaShapedMeasure.cs': {'points': {}}, 'LocalLineShapedMeasure.cs': {'points': {}}, 'LocalRestrictiveOperationZone.cs': {'zoneSubty

#### 1) Development effort
1. LOC written by **property**
2. Number of files that need to be modified by **class**

In [57]:
import re
import pprint

# LOC written by property
for toolitem, metrics in toolitem_metrics_map.items():
    file_path = next((p for p in toolitem_file_paths if p.name == toolitem), None)
    if not file_path:
        continue
        
    with open(file_path, 'r', encoding='Latin-1', errors='ignore') as f:
        lines = f.readlines()
        
    for prop in metrics.keys():
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            if line.startswith('[GenNetCodeSync'):
                loc = 0
                j = i
                prop_found = False
                brace_count = 0
                seen_brace = False
                
                while j < len(lines):
                    curr_line = lines[j].strip()
                    if curr_line and not curr_line.startswith('//'):
                        loc += 1
                        
                        if prop in curr_line:
                            if prop.isalnum():
                                if re.search(rf'\b{prop}\b', curr_line):
                                    prop_found = True
                            else:
                                prop_found = True
                        
                        brace_count += curr_line.count('{')
                        brace_count -= curr_line.count('}')
                        if '{' in curr_line:
                            seen_brace = True
                            
                        ended = False
                        if seen_brace and brace_count == 0:
                            ended = True
                        elif not seen_brace and ';' in curr_line:
                            ended = True
                            
                        if ended:
                            if prop_found:
                                toolitem_metrics_map[toolitem][prop]['LOC'] = loc
                            break
                    j += 1
                
                if prop_found:
                    break
            i += 1

pprint.pprint(toolitem_metrics_map)


{'LocalAbstractEllipse.cs': {'geoData': {'LOC': 4}, 'renderType': {'LOC': 4}},
 'LocalAirspaceDefenseVolume.cs': {'firingPermission': {'LOC': 4},
                                   'overflightPermission': {'LOC': 4}},
 'LocalAirspaceRoute.cs': {'altitudeInMeters': {'LOC': 4},
                           'degrees': {'LOC': 4},
                           'geoCenter': {'LOC': 4},
                           'geoSize': {'LOC': 4},
                           'routeType': {'LOC': 4}},
 'LocalAirspaceTrajectory.cs': {'dependenciesCount': {'LOC': 4},
                                'points': {'LOC': 3}},
 'LocalAirspaceVolume.cs': {'altitudeInMeters': {'LOC': 4},
                            'degrees': {'LOC': 4},
                            'geoCenter': {'LOC': 4},
                            'geoSize': {'LOC': 4},
                            'volumeType': {'LOC': 4}},
 'LocalAreaShapedMeasure.cs': {'points': {'LOC': 4}},
 'LocalArrow.cs': {'arrowData': {'LOC': 4}, 'renderType': {'LOC': 4}},
 'L